### Train a single layer of multi-headed attention to sort digits.

In [1]:
import jax
from jax import Array
import jax.numpy as jnp
from jax.typing import ArrayLike
import numpy as np
import optax
from flax import nnx

In [2]:
n_digits = 5
seq_len = 2 * n_digits  # model input is seq[:-1], e.g. "2 1 7 9 4 | 1 2 4 7"
model_dim = 64
n_heads = 4
batch_size = 256
n_steps = 1000
vocab = "0123456789|"
vocab_size = len(vocab)
sep_id = vocab.index("|")

In [3]:
def get_batch(rng: np.random.Generator, batch_size: int) -> dict[str, Array]:
    """
    Return batch of sorting problems (distinct digits) and a mask of which tokens to predict:

        seq    = 2 1 7 9 4 | 1 2 4 7 9
        is_tgt = 0 0 0 0 0 0 1 1 1 1 1
    """
    x = np.array([rng.permutation(10)[:n_digits] for _ in range(batch_size)])
    sep = np.full((batch_size, 1), sep_id)
    seq = np.concatenate([x, sep, np.sort(x, axis=1)], axis=1)
    is_tgt = np.zeros(seq.shape, dtype=bool)
    is_tgt[:, n_digits + 1:] = True
    return {"seq": jnp.array(seq), "is_tgt": jnp.array(is_tgt)}

In [4]:
def attention(
    query: ArrayLike,
    key: ArrayLike,
    value: ArrayLike,
    mask: ArrayLike,
) -> tuple[Array, Array]:
    """
    Scaled dot-product attention:

        scores = softmax[(Q @ K^T) / sqrt(d_k)]
        attention(q, k, v) = scores @ V

    Shapes:
        query  = (query_len, d_k)
        key    = (key_len,   d_k)
        value  = (value_len, d_v)
        scores = (query_len, key_len)
        mask   = (query_len, key_len)

    Note that in practice, query = key = value for self-attention.
    """
    d_k = query.shape[-1]

    # query and key have shapes (batch, head, seq, model)
    # we want to do the matrix multiplication: (batch, head, seq, model) @ (batch, head, model, seq)
    # the resulting vector has dimension (batch, head, seq, seq)
    # so each head has its own (seq, seq) attention layer, and we run this over batches
    scores = (
        query @ jnp.transpose(key, (0, 1, 3, 2))
    ) / np.sqrt(d_k)

    # We mask to enforce causality or to apply attention to multiple sequences at once.
    scores = jnp.where(mask, scores, -jnp.inf)
    
    # These are the attention weights, which are commonly visualized.
    weights = nnx.softmax(scores)

    # weights has shape (batch, head, seq, seq)
    # value has shape (batch, head, seq, d_k)
    # output has shape (batch, head, seq, d_k)
    return jnp.matmul(weights, value), weights


def make_causal_mask(size: int) -> np.ndarray:
    """
    Creates causality-enforcing decoding mask.
    """
    score_shape = (size, size)
    return np.tril(np.ones(score_shape), k=0).astype(bool)

In [5]:
class MultiheadAttention(nnx.Module):

    def __init__(self, n_heads: int, model_dim: int, rngs):
        """
        Instantiate multi-head attention.
        """
        # Multi-head attention is tiled over the model dim. So if model_dim=64 and n_heads=8, then
        # d_k=8, so the dimension of the queries, keys, and values is 8. If model_dim=128 for 8
        # heads, then the dimension of the queries, keys, and values doubles as well. In principle,
        # I don't know that model_dim has to be a multiple of n_heads, but it makes the
        # implementation cleaner here.
        if model_dim % n_heads != 0:
            raise ValueError(f"{model_dim=} should be divisible by {n_heads=}")

        self.d_k = model_dim // n_heads
        self.n_heads = n_heads
        self.model_dim = model_dim
        self.w_q = nnx.Linear(model_dim, model_dim, rngs=rngs)
        self.w_k = nnx.Linear(model_dim, model_dim, rngs=rngs)
        self.w_v = nnx.Linear(model_dim, model_dim, rngs=rngs)

    def __call__(self, query, key, value, mask):
        """
        Run multi-head attention. Notice there aren't multiple heads anywhere. It's just tiling.
        """
        batch_size, seq_len, model_dim = query.shape

        query = self.w_q(query)
        key = self.w_k(key)
        value = self.w_v(value)

        # Note: d_k * n_heads = model_dim, so:
        # Take model_dim-dimensional vector and split it into n_heads chunks.
        query = query.reshape(batch_size, -1, self.n_heads, self.d_k)
        key = key.reshape(batch_size, -1, self.n_heads, self.d_k)
        value = value.reshape(batch_size, -1, self.n_heads, self.d_k)

        # Before transpose: query, key, and value have shape (batch, seq, head, d_k)
        # After transpose: they have shape (batch, head, seq, d_k)
        query = jnp.transpose(query, (0, 2, 1, 3))
        key = jnp.transpose(key, (0, 2, 1, 3))
        value = jnp.transpose(value, (0, 2, 1, 3))

        # Apply attention across the heads in batch
        x, _ = attention(query, key, value, mask)

        # This effectively concatenates the head outputs. First put back dimensions, then reshape
        # (B, H, S, Dh) -> (B, S, H, Dh)
        x = jnp.transpose(x, (0, 2, 1, 3))
        x = x.reshape(batch_size, seq_len, model_dim)

        return x

In [6]:
class Model(nnx.Module):
    """
    Decoder-only (GPT-style) model with a single layer of multi-headed attention.
    """

    def __init__(
        self, vocab_size: int, seq_len: int, model_dim: int, n_heads: int, rngs: nnx.Rngs
    ):
        """
        Initialize model.
        """
        self.embed = nnx.Embed(vocab_size, model_dim, rngs=rngs)
        self.pe = nnx.Embed(seq_len, model_dim, rngs=rngs)
        self.self_attention = MultiheadAttention(n_heads, model_dim, rngs=rngs)
        self.output_projection = nnx.Linear(model_dim, vocab_size, rngs=rngs)

    def __call__(self, x, mask):
        # Embed tokens and add a learned vector for each position.
        x = self.embed(x) + self.pe(jnp.arange(x.shape[1]))

        # Self-attention, so query = key = value.
        x = self.self_attention(x, x, x, mask)

        # Compute raw logits.
        return self.output_projection(x)

In [7]:
def compute_loss(model: Model, batch: dict[str, ArrayLike]) -> Array:
    """
    Next-token prediction loss, scoring only the target tokens.
    """
    seq_input = batch["seq"][:, :-1]
    seq_output = batch["seq"][:, 1:]
    seq_mask = make_causal_mask(seq_input.shape[1])

    # Compute raw logits.
    logits = model(seq_input, seq_mask)

    # Compute cross-entropy loss.
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, seq_output)

    # Remove first token because unneeded and aligns with seq_output shape
    valid = batch["is_tgt"][:, 1:]
    return (loss * valid).sum() / valid.sum()


@nnx.jit
def train_step(model: Model, optimizer: nnx.Optimizer, batch: dict[str, ArrayLike]) -> Array:
    """
    Jitted training step.
    """
    def loss_fn(model):
        return compute_loss(model, batch)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

In [8]:
model = Model(vocab_size, seq_len, model_dim, n_heads, nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

rng = np.random.default_rng(0)
for step in range(n_steps):
    batch = get_batch(rng, batch_size)
    loss = train_step(model, optimizer, batch)
    if step % 100 == 0:
        print(f"{step=:4d}    loss={loss:.4f}")

Platform 'mps' is experimental and not all JAX functionality may be correctly supported!
[jax-mps] Tip: set JAX_MPS_ASYNC_DISPATCH=1 to opt into async dispatch (experimental; can be much faster for dispatch-bound workloads).


step=   0    loss=2.4094
step= 100    loss=1.1760
step= 200    loss=0.3959
step= 300    loss=0.0206
step= 400    loss=0.0074
step= 500    loss=0.0029
step= 600    loss=0.0017
step= 700    loss=0.0013
step= 800    loss=0.0009
step= 900    loss=0.0006


In [9]:
def greedy_generate(model: Model, seq: ArrayLike) -> Array:
    """
    Given a batch of prompts, greedily (argmax on the logits) generate the sorted digits, one token
    at a time.
    """
    for _ in range(n_digits):
        logits = model(seq, make_causal_mask(seq.shape[1]))
        # pick the max logit, i.e. the next digit
        next_id = jnp.argmax(logits[:, -1, :], axis=-1)
        seq = jnp.concatenate([seq, next_id[:, None]], axis=1)
    return seq


# Evaluate on held out data. A sequence is correct if all of its digits are correct.
batch = get_batch(np.random.default_rng(1), 1000)
prompt = batch["seq"][:, :n_digits + 1]
y_true = batch["seq"]
y_hat = greedy_generate(model, prompt)
acc = (y_hat == y_true).all(axis=1).mean()
print(f"acc={100 * acc:.1f}%")

for seq in y_hat[:5]:
    print(" ".join(vocab[i] for i in seq))

acc=100.0%
8 4 7 0 1 | 0 1 4 7 8
0 1 8 6 5 | 0 1 5 6 8
3 1 5 6 9 | 1 3 5 6 9
0 7 9 3 2 | 0 2 3 7 9
3 1 9 2 8 | 1 2 3 8 9
